# Foundation models on Fold B (GPU)

Full-size runs that the local CPU trial could not do. See `notes/foundation_models.md` for the local results.

**Rules**
- Reads `features_foldB.parquet` only. It holds Fold B train and test rows, marked by the `split` column. Do not point it at the Fold A file.
- All inference runs inside this Colab runtime. Never install or use `tabpfn-client` or any hosted model API, because they upload rows.
- TabPFN v2 only. Versions 2.5 and later need a licence click and a token, so they are left out.
- Writes one small CSV of scores, with no row-level data. Predictions stay in memory.
- Clear outputs before saving a copy anywhere.

Use a GPU runtime (Runtime > Change runtime type). Keep the Drive folder private, and delete the data after the hackathon, as `colab/README.md` says.

## Config

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/flats_private"          # the private Drive folder
FEATURES_DIR = os.environ.get("FLATS_FEATURES_DIR") or DRIVE_DIR
RESULTS_DIR = os.environ.get("FLATS_FM_RESULTS_DIR") or DRIVE_DIR
SMOKE = os.environ.get("FLATS_FM_SMOKE") == "1"              # tiny CPU check of every cell; not a result

SEEDS = [1, 2, 3]
BUDGET_MINUTES = 150     # no new run starts after this
# (model, training rows, ensemble members). Rows 0 means all, about 28,000. Trees run at every size used by another
# model, so each subsample has a fair reference. TabPFN v2 was trained for up to 10,000 rows.
RUNS = [("lgb", 10000, 0), ("cat", 10000, 0), ("xgb", 10000, 0), ("lgb", 0, 0), ("cat", 0, 0), ("xgb", 0, 0),
        ("tabicl", 10000, 4), ("tabicl", 0, 4), ("realmlp", 0, 0), ("tabm", 0, 0), ("tabpfn", 10000, 4)]
TARGETS = ["log_ratio", "rent"]
if os.environ.get("FLATS_FM_RUNS"):
    RUNS = [tuple(r) for r in json.loads(os.environ["FLATS_FM_RUNS"])]
if os.environ.get("FLATS_FM_SEEDS"):
    SEEDS = json.loads(os.environ["FLATS_FM_SEEDS"])
if SMOKE:
    SEEDS, BUDGET_MINUTES, TARGETS = [1], 20, ["log_ratio"]
    RUNS = [("lgb", 300, 0), ("cat", 300, 0), ("xgb", 300, 0), ("tabicl", 300, 1), ("tabpfn", 300, 1), ("realmlp", 300, 0), ("tabm", 300, 0)]
    os.environ.update(FM_EPOCHS="3", FM_TABM_SECONDS="20", TABPFN_ALLOW_CPU_LARGE_DATASET="1")
os.environ.update(TABPFN_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_TELEMETRY="1", DO_NOT_TRACK="1")

## Install and connect

Versions match the local trial. On Colab they have not been run, so a resolver clash here is possible.

In [ ]:
PINS = ["tabpfn==9.2.0", "tabicl==2.2.0", "pytabkit==1.8.0", "tabm==0.0.3", "rtdl_num_embeddings==0.0.12",
        "catboost==1.2.10", "xgboost==3.4.1", "lightgbm==4.7.0"]
try:
    import catboost, lightgbm, pytabkit, rtdl_num_embeddings, tabicl, tabm, tabpfn, xgboost  # noqa: E401,F401
except ImportError:
    # Only ever install on Colab. Elsewhere this could change a shared Python environment.
    if "google.colab" not in sys.modules:
        raise SystemExit("Missing packages. Off Colab, use a separate venv with: " + " ".join(PINS))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINS])

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

assert "tabpfn_client" not in sys.modules, "tabpfn-client is a hosted API and uploads rows"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not on Colab: skipping Drive mount.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THREADS = int(os.environ.get("FLATS_FM_THREADS", os.cpu_count() or 2))
torch.set_num_threads(THREADS)
if "google.colab" in sys.modules:
    assert DEVICE == "cuda", "Choose a GPU runtime first."
print("device", DEVICE, "threads", THREADS)

## Scoring

Copied from `src/validate.py` and `src/models.py`. If those change, copy them again.

In [ ]:
PRED_FLOOR = 100.0
CATEGORICAL = ("suburb", "property_type")
PARKING_PROXIES = {"car_space", "car_spaces", "advert_car_space", "input_car_space"}


def scored_prediction(df: pd.DataFrame) -> pd.DataFrame:
    """Pick the column matching advert parking, else input parking. Drop only when both are unknown."""
    park = df["advert_car_space"].astype(float).fillna(df["input_car_space"].astype(float))
    out = df[park.notna()].copy()
    p = park[park.notna()]
    out["pred"] = np.where(p > 0, out["rent_with_car_space"], out["rent_without_car_space"])
    return out


def score(df: pd.DataFrame) -> dict:
    """df needs: rent, rent_with_car_space, rent_without_car_space, advert_car_space, input_car_space.

    A blank scored prediction counts as an infinite error, worse than any answered miss.
    """
    s = scored_prediction(df)
    err = (s["pred"] - s["rent"]).astype(float)
    abs_err = err.abs().fillna(np.inf)
    ok = err.notna()
    sse = (err[ok] ** 2).sum()
    sst = ((s.loc[ok, "rent"] - s.loc[ok, "rent"].mean()) ** 2).sum()
    return dict(n=len(s), n_excluded=len(df) - len(s), n_blank=int((~ok).sum()), mae=float(abs_err.median()),
                within100=float((abs_err <= 100).mean()), r2=float(1 - sse / sst) if sst else float("nan"),
                bias=float(err.median()))


def matrix(df: pd.DataFrame, cols: list[str], categories: dict) -> pd.DataFrame:
    """Model input. NaN stays NaN; a category unseen in training becomes NaN."""
    X = pd.DataFrame(index=df.index)
    for c in cols:
        if c in categories:
            v = df[c].astype(object)
            X[c] = pd.Categorical(v.where(v.isin(categories[c])), categories=categories[c])
        else:
            X[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    return X


def compose_prices(passes: pd.DataFrame, car_space, premium=None) -> pd.DataFrame:
    """Stage 7 rule. A flat with known parking keeps its own pass and the other column is that minus or plus the
    premium; an unknown flat keeps both passes. premium overrides the two-pass difference where it is not NaN.
    Either way it is floored at $0.
    """
    prem = passes["premium"]
    if premium is not None:
        prem = pd.Series(premium, index=passes.index, dtype=float).fillna(prem)
    prem = prem.clip(lower=0)
    car = pd.Series(car_space, index=passes.index, dtype=float)
    w, wo = passes["pred_with"], passes["pred_without"]
    without = (w - prem).where(car > 0, wo).clip(lower=PRED_FLOOR)
    with_ = (wo + prem).where(car == 0, w)
    return pd.DataFrame({"rent_with_car_space": np.maximum(with_, without), "rent_without_car_space": without})

## Load Fold B

In [ ]:
path = Path(FEATURES_DIR) / "features_foldB.parquet"
assert path.exists(), f"Missing {path}. Upload it from out/real/."
meta = pq.read_schema(path).metadata or {}
assert meta.get(b"flats_source") == b"real" or SMOKE, "not a real-data file"
df = pd.read_parquet(path)
df = df.assign(target_date=pd.to_datetime(df.target_date))
CUTOFF, END = pd.Timestamp("2026-03-31"), pd.Timestamp("2026-05-31")
train_all, test = df[df.split == "train"], df[df.split == "test"]
train_all = train_all[train_all.best_anchor > 0]
assert train_all.target_date.max() <= CUTOFF, "train target after the fold cutoff"
assert test.target_date.between("2026-04-01", END).all(), "test row outside Fold B"
cols = [c for c in json.loads(meta[b"flats_feature_cols"]) if c not in PARKING_PROXIES]
if "car_space_resolved" not in cols:
    cols.append("car_space_resolved")
if SMOKE:
    train_all = train_all.sample(min(3000, len(train_all)), random_state=0)
    test = test.sample(min(120, len(test)), random_state=0)
first_time = test.anchor_own.isna().to_numpy()
print(len(train_all), "train rows,", len(test), "test rows,", len(cols), "features")

## Models

Copied from `local/run_foundation.py`, which is what the local trial ran.

In [ ]:
class M:
    """The few src.models names the copied functions use."""
    CATEGORICAL = CATEGORICAL
    LGB_PARAMS = dict(objective="l1", learning_rate=0.05, num_leaves=63, min_data_in_leaf=20, feature_fraction=0.8,
                      bagging_fraction=0.8, bagging_freq=1, cat_smooth=10, verbosity=-1, num_threads=4,
                      deterministic=True, force_row_wise=True)
    NUM_BOOST_ROUND = 400
    _matrix = staticmethod(matrix)
    compose_prices = staticmethod(compose_prices)


def subsample(train: pd.DataFrame, n: int, seed: int, how: str = "random") -> pd.DataFrame:
    """The training rows for this seed. Every model given the same n, seed and how gets the same rows."""
    if not n or n >= len(train):
        return train
    if how == "recent":
        return train.sort_values("target_date", kind="stable").tail(n)
    return train.iloc[np.sort(np.random.default_rng(1000 + seed).choice(len(train), n, replace=False))]


def design(train: pd.DataFrame, test: pd.DataFrame, cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Model input in the shape src.models uses, with test stacked as car=1 rows then car=0 rows."""
    cats = {c: sorted(train[c].dropna().astype(str).unique()) for c in cols
            if c in M.CATEGORICAL or not pd.api.types.is_numeric_dtype(train[c])}
    Xtr, Xte = M._matrix(train, cols, cats), M._matrix(test, cols, cats)
    both = pd.concat([Xte.assign(car_space_resolved=1.0), Xte.assign(car_space_resolved=0.0)], ignore_index=True)
    return Xtr, both, list(cats)


def ordinal(Xtr: pd.DataFrame, Xte: pd.DataFrame, cats: list[str]) -> tuple[np.ndarray, np.ndarray, list[int]]:
    """Float arrays for models that take a plain matrix. Category codes, with NaN for blank."""
    def enc(X):
        X = X.copy()
        for c in cats:
            X[c] = X[c].cat.codes.astype(float).replace(-1, np.nan)
        return X.to_numpy(np.float32)
    return enc(Xtr), enc(Xte), [Xtr.columns.get_loc(c) for c in cats]


def impute(Xtr: pd.DataFrame, Xte: pd.DataFrame, cats: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """For models that refuse NaN: training medians plus a flag per blank column. Blank categories become their own level."""
    med = Xtr.drop(columns=cats).median()
    flags = [c for c in med.index if Xtr[c].isna().any()]

    def fill(X):
        out = X.assign(**{f"{c}_blank": X[c].isna().astype(float) for c in flags}).fillna(med)
        for c in cats:
            out[c] = out[c].cat.add_categories("NA").fillna("NA")
        return out
    return fill(Xtr), fill(Xte)


def target(train: pd.DataFrame, form: str) -> np.ndarray:
    rent = train.rent.to_numpy(float)
    return np.log(rent / train.best_anchor.to_numpy(float)) if form == "log_ratio" else rent


def to_rent(raw: np.ndarray, test: pd.DataFrame, form: str) -> tuple[np.ndarray, np.ndarray]:
    """Split the stacked predictions into (pred_with, pred_without) in weekly dollars."""
    w, wo = raw[:len(test)], raw[len(test):]
    if form == "log_ratio":
        a = test.best_anchor.to_numpy(float)
        return a * np.exp(w), a * np.exp(wo)
    return w, wo


def m_lgb(Xtr, y, Xte, cats, seed, threads, **_):
    import lightgbm as lgb
    params = {**M.LGB_PARAMS, "seed": seed, "num_threads": threads}
    b = lgb.train(params, lgb.Dataset(Xtr, y, categorical_feature=cats), num_boost_round=M.NUM_BOOST_ROUND)
    return {"": b.predict(Xte)}


def m_cat(Xtr, y, Xte, cats, seed, threads, **_):
    from catboost import CatBoostRegressor

    def prep(X):
        X = X.copy()
        for c in cats:
            X[c] = X[c].astype(object).where(X[c].notna(), "NA").astype(str)
        return X
    m = CatBoostRegressor(loss_function="MAE", iterations=800, learning_rate=0.05, depth=6, random_seed=seed,
                          thread_count=threads, verbose=False, allow_writing_files=False, cat_features=cats)
    m.fit(prep(Xtr), y)
    return {"": m.predict(prep(Xte))}


def m_xgb(Xtr, y, Xte, cats, seed, threads, **_):
    import xgboost as xgb
    params = dict(objective="reg:absoluteerror", tree_method="hist", eta=0.05, max_depth=7, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=5, seed=seed, nthread=threads, verbosity=0)
    b = xgb.train(params, xgb.DMatrix(Xtr, y, enable_categorical=True), num_boost_round=M.NUM_BOOST_ROUND)
    return {"": b.predict(xgb.DMatrix(Xte, enable_categorical=True))}


def m_tabpfn(Xtr, y, Xte, cats, seed, threads, n_estimators=2, version="v2", **_):
    import torch
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion
    torch.set_num_threads(threads)
    A, B, ci = ordinal(Xtr, Xte, cats)
    make = TabPFNRegressor.create_default_for_version
    m = make(ModelVersion.V2, n_estimators=n_estimators, device=DEVICE, random_state=seed,
             ignore_pretraining_limits=True, categorical_features_indices=ci, memory_saving_mode=True)
    m.fit(A, y)
    p = m.predict(B, output_type="main")
    return {"mean": p["mean"], "median": p["median"]}


def m_tabicl(Xtr, y, Xte, cats, seed, threads, n_estimators=4, **_):
    import torch
    from tabicl import TabICLRegressor
    torch.set_num_threads(threads)
    A, B, _ci = ordinal(Xtr, Xte, cats)
    m = TabICLRegressor(n_estimators=n_estimators, device=DEVICE, random_state=seed, n_jobs=threads)
    m.fit(A, y)
    p = m.predict(B, output_type=["mean", "median"])
    return {"mean": np.asarray(p["mean"]), "median": np.asarray(p["median"])}


def m_realmlp(Xtr, y, Xte, cats, seed, threads, **_):
    import torch
    from pytabkit import RealMLP_TD_Regressor
    torch.set_num_threads(threads)
    kw = dict(device=DEVICE, random_state=seed, n_cv=1, n_refit=0, verbosity=0, n_threads=threads)
    m = RealMLP_TD_Regressor(**kw, n_epochs=int(os.environ.get("FM_EPOCHS", 128)), val_metric_name="mae",
                             train_metric_name=os.environ.get("FM_REALMLP_LOSS", "pinball(0.5)"))
    Xtr, Xte = impute(Xtr, Xte, cats)
    m.fit(Xtr, y)
    return {"": np.asarray(m.predict(Xte)).reshape(-1)}


def m_tabm(Xtr, y, Xte, cats, seed, threads, **_):
    """TabM-mini with piecewise-linear embeddings, trained on L1 per ensemble member. The median over members is the output."""
    import torch
    import torch.nn.functional as F
    from rtdl_num_embeddings import PiecewiseLinearEmbeddings, compute_bins
    from sklearn.preprocessing import QuantileTransformer
    from tabm import TabM

    torch.set_num_threads(threads)
    torch.manual_seed(seed)
    A, B, ci = ordinal(Xtr, Xte, cats)
    nums = [i for i in range(A.shape[1]) if i not in ci]
    rng = np.random.default_rng(seed)
    is_na = np.isnan(A[:, nums]).mean(axis=0) > 0
    fill = np.nanmedian(A[:, nums], axis=0)
    def num(X):
        Z = X[:, nums]
        miss = np.isnan(Z)[:, is_na].astype(np.float32)
        return np.where(np.isnan(Z), fill, Z), miss
    Zt, Mt = num(A)
    Zv, Mv = num(B)
    qt = QuantileTransformer(n_quantiles=min(1000, len(Zt)), output_distribution="normal", subsample=None, random_state=seed).fit(
        Zt + rng.normal(0, 1e-5, Zt.shape) * Zt.std(axis=0, keepdims=True))
    Zt, Zv = np.c_[qt.transform(Zt), Mt].astype(np.float32), np.c_[qt.transform(Zv), Mv].astype(np.float32)
    card = [int(np.nanmax(A[:, i])) + 2 for i in ci]
    def catc(X):
        return torch.as_tensor(np.nan_to_num(X[:, ci], nan=-1).clip(-1, np.array(card) - 3).astype(np.int64) + 1)
    Ct, Cv = catc(A), catc(B)
    mu, sd = np.median(y), np.subtract(*np.percentile(y, [75, 25])) / 1.349
    yt = torch.as_tensor((y - mu) / sd, dtype=torch.float32)
    idx = rng.permutation(len(y))
    n_val = max(200, len(y) // 10)
    va, tr = torch.as_tensor(idx[:n_val]), torch.as_tensor(idx[n_val:])
    part = Zt[idx[n_val:]]
    keep = part.max(axis=0) > part.min(axis=0)   # a near-constant column can be constant in one split, and that breaks the bins
    Zt, Zv = torch.as_tensor(Zt[:, keep]), torch.as_tensor(Zv[:, keep])
    emb = PiecewiseLinearEmbeddings(compute_bins(Zt[tr], n_bins=24), d_embedding=16, activation=False, version="B")
    Zt, Zv, Ct, Cv, yt = (t.to(DEVICE) for t in (Zt, Zv, Ct, Cv, yt))
    net = TabM.make(n_num_features=Zt.shape[1], cat_cardinalities=card, d_out=1, num_embeddings=emb, k=16, n_blocks=2,
                    d_block=256, dropout=0.1, arch_type="tabm-mini").to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=3e-4)

    def run(Z, Cc):
        with torch.no_grad():
            return torch.cat([net(Z[i:i + 2048], Cc[i:i + 2048]).squeeze(-1) for i in range(0, len(Z), 2048)])
    best, best_state, bad, started = np.inf, None, 0, time.time()
    budget = float(os.environ.get("FM_TABM_SECONDS", 420))
    for epoch in range(100):
        net.train()
        for b in torch.as_tensor(rng.permutation(len(tr))).split(256):
            rows = tr[b]
            pred = net(Zt[rows], Ct[rows]).squeeze(-1)
            loss = F.l1_loss(pred, yt[rows][:, None].expand_as(pred))
            opt.zero_grad()
            loss.backward()
            opt.step()
        net.eval()
        v = (run(Zt[va], Ct[va]).median(dim=1).values - yt[va]).abs().mean().item()
        if v < best - 1e-5:
            best, bad = v, 0
            best_state = {k: t.clone() for k, t in net.state_dict().items()}
        else:
            bad += 1
        if bad >= 8 or time.time() - started > budget:
            break
    print(f"tabm epochs={epoch + 1} val_l1={best:.4f}", flush=True)
    net.load_state_dict(best_state)
    net.eval()
    return {"": run(Zv, Cv).median(dim=1).values.cpu().numpy() * sd + mu}


def needed_rows(test: pd.DataFrame) -> np.ndarray:
    """Mask over the stacked car=1, car=0 rows: the passes that can reach the score.

    A row whose advert parking matches its input parking is scored on its own pass alone. Only flipped or blank
    rows need both. This only saves compute on slow models. LightGBM and CatBoost scored identically with and without it.
    """
    inp = test.input_car_space.astype(float).to_numpy()
    adv = test.advert_car_space.astype(float).fillna(test.input_car_space.astype(float)).to_numpy()
    both = ~((inp > 0) == (adv > 0)) | np.isnan(inp) | np.isnan(adv)
    return np.r_[(adv > 0) | both, ~(adv > 0) | both]


def fill_passes(raw: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Put predictions for needed rows back in the stack. A skipped pass copies the row's other pass."""
    full = np.full(len(mask), np.nan)
    full[mask] = raw
    h = len(mask) // 2
    w, wo = full[:h], full[h:]
    return np.r_[np.where(np.isnan(w), wo, w), np.where(np.isnan(wo), w, wo)]


def scored_frame(test: pd.DataFrame, w: np.ndarray, wo: np.ndarray) -> pd.DataFrame:
    p = pd.DataFrame({"pred_with": w, "pred_without": wo, "premium": w - wo}, index=test.index)
    prices = M.compose_prices(p, test["input_car_space"])
    return pd.concat([test[["rent", "advert_car_space", "input_car_space"]], prices], axis=1)


def boot_gap(ea: np.ndarray, eb: np.ndarray, reps: int = 2000, seed: int = 11) -> tuple[float, float, float]:
    """Paired bootstrap over test rows of mean-over-seeds median error A minus B. ea, eb are (seeds, rows)."""
    rng = np.random.default_rng(seed)
    n = ea.shape[1]
    gaps = np.empty(reps)
    for r in range(reps):
        i = rng.integers(0, n, n)
        gaps[r] = np.median(ea[:, i], axis=1).mean() - np.median(eb[:, i], axis=1).mean()
    return float(np.median(ea, axis=1).mean() - np.median(eb, axis=1).mean()), *np.percentile(gaps, [2.5, 97.5]).tolist()


MODELS = dict(lgb=m_lgb, cat=m_cat, xgb=m_xgb, tabpfn=m_tabpfn, tabicl=m_tabicl, realmlp=m_realmlp, tabm=m_tabm)

## Run

Each finished run rewrites the results CSV, so a dropped session keeps what it finished. A failed run is recorded and the rest carry on.

In [ ]:
def errors(w, wo):
    s = scored_prediction(scored_frame(test, w, wo))
    return (s["pred"] - s["rent"]).abs().fillna(np.inf)


results, preds, started = [], {}, time.time()
out_csv = Path(RESULTS_DIR) / "foundation_results.csv"
mask = needed_rows(test)   # skips passes that cannot reach the score; the score does not change
for name, n, members in RUNS:
    for form in TARGETS:
        for seed in SEEDS:
            if (time.time() - started) / 60 > BUDGET_MINUTES:
                print("budget reached, skipping", name, n, form, seed)
                continue
            t0 = time.time()
            try:
                train = subsample(train_all, n, seed)
                Xtr, Xte, cats = design(train, test, cols)
                out = MODELS[name](Xtr, target(train, form), Xte[mask].reset_index(drop=True), cats, seed, THREADS, n_estimators=members)
            except Exception as e:  # one failed run must not stop the queue
                results.append(dict(model=name, n=n, target=form, seed=seed, error=repr(e)[:200]))
                print("FAILED", name, n, form, seed, repr(e)[:200])
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()
                continue
            for variant, raw in out.items():
                label = name + (f"-{variant}" if variant else "")
                w, wo = to_rent(fill_passes(np.asarray(raw, float), mask), test, form)
                preds[(label, n, form, seed)] = (w, wo)
                s = score(scored_frame(test, w, wo))
                f = score(scored_frame(test[first_time], w[first_time], wo[first_time]))
                results.append(dict(model=label, n=n, target=form, seed=seed, mae=s["mae"], within100=s["within100"], bias=s["bias"],
                                    first_time_mae=f["mae"], first_time_within100=f["within100"], seconds=round(time.time() - t0, 1)))
                print(label, n, form, seed, round(s["mae"], 2), round(s["within100"], 3), f"{time.time() - t0:.0f}s")
            pd.DataFrame(results).to_csv(out_csv, index=False)

## Compare

Seed means, then the paired bootstrap gap against LightGBM on log(rent / anchor) at the same training size. A negative gap means better than LightGBM. Ignore gaps under $1 unless the interval is clearly clear of zero.

In [ ]:
table = pd.DataFrame(results)
ok = table[table.mae.notna()] if "mae" in table else table.iloc[:0]
print(ok.groupby(["n", "model", "target"])[["mae", "within100", "first_time_mae", "first_time_within100", "seconds"]].mean().round(3).sort_values(["n", "mae"]).to_string())


def seed_errors(label, n, form):
    return {s: errors(*p).to_numpy() for (m, nn, f, s), p in preds.items() if (m, nn, f) == (label, n, form)}


gaps = []
for label, n, form in sorted({k[:3] for k in preds}):
    ref, mine = seed_errors("lgb", n, "log_ratio"), seed_errors(label, n, form)
    common = sorted(set(ref) & set(mine))
    if common and (label, form) != ("lgb", "log_ratio"):
        g, lo, hi = boot_gap(np.array([mine[s] for s in common]), np.array([ref[s] for s in common]), reps=1000)
        gaps.append(dict(model=label, n=n, target=form, gap=g, lo=lo, hi=hi))
gaps = pd.DataFrame(gaps)
print(gaps.round(2).to_string(index=False) if len(gaps) else "no gaps")

## 50/50 blend with LightGBM

A fixed weight, so nothing is tuned on the test rows. It is computed for each foundation or deep model that ran on all rows or the largest size.

In [ ]:
blend = []
for label, n, form in sorted({k[:3] for k in preds}):
    if label.split("-")[0] in ("lgb",):
        continue
    for s in SEEDS:
        a, b = preds.get((label, n, form, s)), preds.get(("lgb", n, "log_ratio", s))
        if a and b:
            w, wo = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
            r = score(scored_frame(test, w, wo))
            blend.append(dict(model=label, n=n, target=form, seed=s, mae=r["mae"], within100=r["within100"]))
blend = pd.DataFrame(blend)
if len(blend):
    print(blend.groupby(["n", "model", "target"])[["mae", "within100"]].mean().round(3).sort_values(["n", "mae"]).to_string())
    blend.to_csv(Path(RESULTS_DIR) / "foundation_blend.csv", index=False)